# 🫀 실험 17 — **4전극 웨어러블**: 두 평면을 한 번에 덮으면 12유도에 얼마나 붙나

**MedKOS / `notebooks/exp17_wearable_configs.ipynb`** · 퀘스트 `ailab-2026-0015`
**다운로드 없음** · 실험16 arm **재사용** · 새로 학습하는 것은 **45회 (~45분)**

---

## 실험16 이 남긴 자리

실험16 이 보인 것은 **평면이 유도를 고른다**는 것이었다:

| | 사지 `{I,II}` 가 이김 | 가슴 `{II,V1}` 이 이김 |
|---|---|---|
| 부위 | IMI · LMI · ALMI · ILMI | ASMI · AMI · IPLMI |
| 최선 잔여 결손 D | −0.002 ~ +0.043 | +0.039 ~ +0.065 |

**소견별로 최선을 고르면 D 는 최대 0.065 밖에 안 된다.** 그런데 배포할 때는
**어느 부위인지 모른 채** 전극을 붙여야 한다. 소견을 알면 유도를 고를 수 있다는 건
쓸모없는 얘기다 — 유도를 먼저 정하고 소견을 맞혀야 한다.

> **그래서 두 평면을 한꺼번에 덮으면 되지 않나?** 그게 이 실험이다.

## 전극 수를 정직하게 센다 (실험16 에서 틀렸던 부분)

| 구성 | 유도 수 | **전극 수** | 실제 기기 |
|---|---|---|---|
| `{II}` | 1 | 2 | 단일유도 패치 |
| `{I,II}` | 2 | **3** (RA·LA·LL) | 3전극 홀터 — 전두면 6유도 전부 복원 |
| `{II,V1}` | 2 | **4** (+C1) | V1 은 Wilson 중심단자가 필요하다 |
| **`{I,II,V1}`** | **3** | **4** (RA·LA·LL·C1) | ★ 이번 주인공 — 두 평면 |
| `{I,II,V2}` | 3 | 4 | V2 가 전벽에 더 낫다는 교과서 주장 검정 |
| `{I,II,V1,V5}` | 4 | 5 | 측벽 흉부까지 |
| `{12}` | 12 | 10 | 표준 심전도 |

`{II,V1}` 도 이미 **4전극**이다(Wilson 단자에 RA·LA·LL 이 다 필요하다).
그렇다면 **같은 4전극으로 `{I,II,V1}` 을 못 쓸 이유가 없다** — I 를 버릴 이유가 없다.
실험16 의 `{II,V1}` 은 대조군으로서는 옳았지만 **배포 후보로서는 열등**하다.

## 왜 V2 도 보는가 — 순환기 수업

전중격경색은 **V1–V3** 에서 읽는데, 그 중 신호가 가장 큰 곳은 보통 **V2** 다
(정상에서 R 파가 가장 급격히 커지는 자리라 R 파 감소가 눈에 띈다).
V1 은 오히려 정상에서도 QS 패턴이 흔해 위양성이 많다.

> **전벽을 하나로 커버한다면 V1 이냐 V2 냐** — 4전극 기기 설계에서 실제로 정해야 하는 값이다.

## 무엇을 얼마나 돌리나 — 실험16 arm 을 **재사용한다**

실험16 의 G0-b 가 `ΔAUROC = 0.0000 (14/14)` 로 **학습이 결정론적**임을 확인했다.
그래서 실험16 의 4구성 × 3시드 × 5겹 = 60개 arm 을 **그대로 읽는다.**

| | 학습 |
|---|---|
| 【G0-c】 재사용 정합성 대조 (`{I,II}` 시드0) | 5 |
| `{I,II,V1}` × 3시드 × 5겹 | 15 |
| `{I,II,V2}` × 3시드 × 5겹 | 15 |
| `{I,II,V1,V5}` × 3시드 × 5겹 | 15 |
| **합** | **50회** |

소요 시간은 GPU 에 달렸다. 1회 학습이 관측된 값은 **약 55초**(실험15c 40회/2,125초,
실험15d 30회/1,674초)이고, 노트북이 **G0-c 5회로 이번 세션의 실제 속도를 재서**
남은 시간을 찍어준다.

| GPU | 1회 | 50회 |
|---|---|---|
| L4 (관측) | ~55s | **~46분** |
| T4 (추정 1.5~1.8배) | ~85~100s | **~70~85분** |

> T4 가 사양상 FP32 는 L4 의 1/3.7 이지만(8.1 vs 30.3 TFLOPS) 이 모델은 작아서
> **연산이 아니라 데이터 복사·커널 오버헤드가 지배한다**(이론 연산량은 겹당 몇 초인데
> 실측은 55초다). 그래서 체감 차이는 1.5~1.8배에 그친다. 메모리 대역폭은 오히려
> T4 가 조금 높다(320 vs 300 GB/s).
>
> ⚠️ **속도를 위해 mixed_float16 을 켜면 안 된다.** 수치가 달라져 실험15·16 arm 과의
> 비교가 깨진다("한 번에 하나만" 원칙 + 재사용 전제).

### ⚠️ 재사용의 함정 — 【G0-c】를 넣은 이유

`P-1` 은 **새로 학습한** `{I,II,V1}` 을 **재사용한** `{I,II}`·`{II,V1}` 과 직접 비교한다.
**GPU 나 라이브러리가 실험16 과 다르면 한쪽 팔만 다른 환경**이 되어, 그 차이가 결론
위에 그대로 얹힌다 — 실험16 에서 60회를 다 새로 돌려 피했던 바로 그 함정이다.

그래서 `{I,II}` 시드0 를 **5겹만 다시 학습해 대조**한다(5회 ≈ 5~9분).
`max |ΔAUROC| ≤ 0.02` 면 재사용을 확정하고, 넘으면 `ON_DRIFT="retrain"` 이
**실험16 구성 60회도 이 세션에서 다시 돌린다**(총 110회). 두 GPU 의 manifest 도 찍어준다.

## 사전등록 (결과 보기 전에 고정)

판정은 **시드 수준 t-CI(df=2)** — 실험15d 규약.
지표는 AUROC 기반 `D(c) = AUROC({12}) − AUROC(c)` 와 **경보율**(민감도 0.90) 둘 다.

| | 예측 | 무엇이 걸려 있나 |
|---|---|---|
| **G0** | 유도 순서 항등식 + 실험16 arm 재사용 정합성(부위·정렬 일치) | 재사용의 전제 |
| **P-1** | **합집합**: 모든 부위에서 `D({I,II,V1}) ≤ min(D({I,II}), D({II,V1})) + 0.01` | 모델이 두 평면을 **합칠** 수 있는가, 아니면 맞바꾸는가 |
| **P-2 ★★** | **배포 관문**: 7부위 전부에서 `D({I,II,V1}) < 0.05` | 4전극이 12유도의 0.05 AUROC 이내인가 |
| **P-3 ★** | **경보 관문**: 7부위 **경보율** 중앙값이 `{12}` 의 **1.25배 이내** | 실험16 이 배운 것 — AUROC 가 붙어도 경보율은 벌어진다 |
| **P-4** | **V2 우위**: 횡단면(ASMI·AMI)에서 `D({I,II,V2}) < D({I,II,V1})` | 전벽 커버 유도를 V1 로 할 것인가 V2 로 할 것인가 |
| **P-5** | **V5 추가 이득**: `D({I,II,V1,V5}) < D({I,II,V1})` 이 측벽(LMI·ALMI)에서 성립 | 5번째 전극을 붙일 값어치가 있는가 |

**붕괴 감시**는 실험15·16 과 같은 계약(부위별, 과반 사망 시 무효).

> ⚠️ **P-1 은 실패할 수 있고, 실패가 곧 발견이다.** 실험16 에서 이미 `ILMI` 가
> "V1 을 더했더니 나빠지는" 위반을 냈고, `LMI`·`IMI` 는 `{12}` 보다 `{I,II}` 가 나았다.
> **유도를 더하면 용량이 나뉜다**는 현상이 실재한다. `{I,II,V1}` 이 두 2유도 구성의
> 좋은 쪽에 못 미치면, 답은 "유도를 더 붙여라"가 아니라 **"구조를 바꿔라"**(실험18) 다.

## 이 실험이 무엇을 정하나

- `P-2`·`P-3` ✅ → **4전극 웨어러블 사양이 확정된다.** 차별점 A 의 최종 산출물이고
  외부검증(Chapman/Ningbo)으로 바로 넘어갈 수 있다.
- `P-1` ❌ → 용량 경쟁이 확인된 것이므로 **실험18(유도 attention·평면 분리 헤드)** 로 간다.
- `P-4` 로 **V1/V2 를 확정**하고, `P-5` 로 5번째 전극의 값어치를 판단한다.


In [ ]:
# CELL 0 — 공용 사전점검 (pipelines/ecg_preflight.py 인라인)
class LabelVocabError(ValueError):
    pass

# ── pipelines/ecg_preflight.py 인라인 (원본·테스트는 repo)
def assert_label_vocab(requested, available, kind="label", counts=None, min_count=1):
    """요청한 이름이 실제 어휘에 **전부** 있는지 확인한다. 하나라도 없으면 예외.

    0건은 "데이터에 그 소견이 없다"가 아니라 **대개 이름을 잘못 골랐다**는 뜻이다.
    그 둘을 구별하려고 어휘 자체를 대조한다.

    requested : 쓰려는 이름들
    available : 데이터에서 실제로 관측된 이름 집합
    counts    : {이름: 건수} (있으면 min_count 미만도 함께 보고)
    """
    requested, available = list(requested), set(available)
    unknown = [r for r in requested if r not in available]
    if unknown:
        raise LabelVocabError(
            f"{kind} 어휘에 없는 이름 {unknown}.\n"
            f"  → 0건이 나온 이유는 '데이터에 없어서'가 아니라 **이름이 틀려서**다.\n"
            f"  실제 어휘({len(available)}개): {sorted(available)}"
        )
    thin = []
    if counts:
        thin = [(r, counts.get(r, 0)) for r in requested if counts.get(r, 0) < min_count]
    return {"ok": True, "n_requested": len(requested), "thin": thin}

def decide(lo, hi, thr, direction):
    """사전등록 관문의 **유일한** 계약: 지지(True) / 기각(False) / 미결(None).

    CI 가 임계값을 걸치면 **기각이 아니라 미결**이다. 검정력 부족을 반증으로
    위장하지 않기 위해서다. 점추정 2분 채점은 금지한다(실험13b·14 에서 그 실수를 했다).
    """
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr:
            return True
        if hi < thr:
            return False
    else:
        if hi < thr:
            return True
        if lo > thr:
            return False
    return None

MARK = {True: "✅ 지지", False: "❌ 기각", None: "⚠️ 미결"}

def collapse_report(scores, floors, names, lift=1.2):
    """붕괴 감시 — 단위(클래스·부위)별로 판정한다.

    scores : {이름: 성능}   floors : {이름: 무작위 수준(유병률 등)}
    성능이 floor 의 `lift` 배도 안 되면 그 단위는 죽은 것으로 본다.

    ★ 하나 죽었다고 실험 전체를 무효화하지 않는다. 무효 판단은 호출자가
      '과반이 죽었는가 / 대조군 한쪽이 통째로 죽었는가'로 따로 한다.
    """
    dead = [n for n in names if scores.get(n, 0.0) < floors.get(n, 0.0) * lift]
    alive = [n for n in names if n not in dead]
    return {"dead": dead, "alive": alive,
            "fatal_majority": len(dead) >= len(names) / 2 if names else True}

def assert_arm_shape(arm, expected_rows, name="arm"):
    """저장된 arm 의 행 수가 **겹 크기**인지 확인한다.

    MedKOSRun.save_arm 은 그 겹의 예측만 저장한다(전체 길이가 아니다).
    겹 순서는 `np.where(CV == k)[0]` 의 오름차순이므로,
      OOF[np.where(CV == k)[0]] = load_arm(...)      ← 이렇게 **넣는다**
      load_arm(...)[전역인덱스]                       ← 이렇게 자르면 IndexError
    실험15d G0 에서 이 혼동으로 터졌다.
    """
    n = arm.shape[0]
    if n != expected_rows:
        raise ValueError(
            f"{name} 행 수 {n} != 기대 {expected_rows}.\n"
            "  → arm 은 **겹 크기**로 저장된다. 전역 인덱스로 자르지 말고 "
            "OOF[np.where(CV==k)[0]] = arm 형태로 넣을 것."
        )
    return {"ok": True, "rows": n}

FRONTAL_IDENTITIES = (
    ("III", 2, lambda I, II: II - I),                 # 아인트호벤
    ("aVR", 3, lambda I, II: -(I + II) / 2.0),        # 골드버거
    ("aVL", 4, lambda I, II: I - II / 2.0),
    ("aVF", 5, lambda I, II: II - I / 2.0),
)

def assert_lead_order(X, tol=0.02, sample=200, seed=0):
    """12유도 캐시의 **채널 순서**를 신호 자체로 검증한다.

    헤더의 유도 이름을 믿지 말고 아인트호벤·골드버거 항등식으로 확인한다:
        III = II − I,  aVR = −(I+II)/2,  aVL = I − II/2,  aVF = II − I/2
    넷이 모두 맞으면 0..5 = I,II,III,aVR,aVL,aVF 이고 표준 순서상 6..11 = V1..V6 이다.
    → `{I,II}` = [0,1], `{II,V1}` = [1,6] 을 쓸 근거가 생긴다.

    유도 순서를 틀리면 **예외 없이 조용히 다른 실험**이 된다. 그래서 잰다.
    ※ 원신호(mV) 전제 — 채널별로 정규화한 배열에는 쓸 수 없다.
    """
    import numpy as np
    if X.ndim != 3 or X.shape[2] != 12:
        raise ValueError(f"X 는 (n, t, 12) 여야 한다 — 받은 모양 {X.shape}")
    rs = np.random.RandomState(seed)
    idx = rs.choice(len(X), size=min(sample, len(X)), replace=False)
    S = X[idx].astype("float64")
    I, II = S[:, :, 0], S[:, :, 1]
    report, bad = {}, []
    for name, j, f in FRONTAL_IDENTITIES:
        want = f(I, II)
        scale = np.abs(want).mean() + 1e-9
        err = float(np.abs(S[:, :, j] - want).mean() / scale)
        report[name] = err
        if err > tol:
            bad.append(f"{name}(ch{j}) 상대오차 {err:.3f}")
    if bad:
        raise ValueError(
            "유도 순서가 표준(I,II,III,aVR,aVL,aVF,V1..V6)이 아니다: " + ", ".join(bad) + "\n"
            "  → 항등식이 깨졌다는 것은 채널 배치가 다르거나 채널별 정규화가 걸렸다는 뜻이다.\n"
            "  마스크 인덱스([0,1] 사지 · [1,6] II+V1)를 그대로 쓰면 조용히 다른 실험이 된다."
        )
    return {"ok": True, "n_checked": len(idx), "rel_err": report}

print("사전점검 적재: assert_label_vocab · decide · collapse_report · "
      "assert_arm_shape · assert_lead_order")

In [ ]:
# CELL 1 — 설정 + 실험16 arm 연결(재사용)
!pip -q install wfdb

import os, sys, json, time, ast, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ★★ 실험15·15c·15d·16 과 한 글자도 달라선 안 되는 블록
K_FOLD, EPOCHS, SEED0, BOOT, NMIN = 5, 20, 20260801, 2000, 50
SITE_CANDIDATES = ["IMI", "ILMI", "IPMI", "IPLMI", "ASMI", "AMI", "ALMI", "LMI", "PMI"]
SITE_PLANE = {"IMI": "전두면", "ILMI": "혼합", "IPMI": "혼합", "IPLMI": "혼합",
              "ASMI": "횡단면", "AMI": "횡단면", "ALMI": "혼합",
              "LMI": "혼합", "PMI": "횡단면"}
SEEDS = [0, 1, 2]
# ★★ 여기까지

# 유도 인덱스: 0=I 1=II 2=III 3=aVR 4=aVL 5=aVF 6=V1 7=V2 8=V3 9=V4 10=V5 11=V6
#              (CELL 2 의 아인트호벤 항등식 검증을 통과해야 유효하다)
OLD_CFG = {"II": [1], "I+II": [0, 1], "II+V1": [1, 6], "12": list(range(12))}   # 실험16 재사용
NEW_CFG = {"I+II+V1":    [0, 1, 6],           # ★ 4전극 — 두 평면
           "I+II+V2":    [0, 1, 7],           # 전벽 커버를 V2 로
           "I+II+V1+V5": [0, 1, 6, 10]}       # 5전극 — 측벽 흉부까지
CONFIGS = {**OLD_CFG, **NEW_CFG}
MAIN     = "I+II+V1"      # 배포 후보
REF      = "12"
D_THR    = 0.05           # P-2 배포 관문(AUROC)
ALARM_R  = 1.25           # P-3 경보 관문(12유도 대비 배수)
UNION_EPS = 0.01          # P-1 허용 오차

# ── 재사용 안전장치. 실험16 arm 은 **다른 세션·다른 GPU**에서 나온 것일 수 있다.
#    P-1 은 새로 학습한 {I,II,V1} 을 재사용한 {I,II}·{II,V1} 과 직접 비교하므로,
#    환경이 바뀌면 그 차이가 결론 위에 그대로 얹힌다(실험16 에서 피했던 바로 그 함정).
#    → G0-c 로 {I,II} 시드0 를 5겹만 다시 학습해 대조한다(5회 ≈ 5~9분).
DRIFT_THR = 0.02          # 최대 |ΔAUROC| 가 이걸 넘으면 재사용을 신뢰하지 않는다
ON_DRIFT  = "retrain"     # "retrain"(권장) | "stop" | "continue"

# 전극 수(정직하게) — V1·V2·V5 는 Wilson 중심단자가 필요해 RA·LA·LL 을 항상 포함한다
N_ELEC = {"II": 2, "I+II": 3, "II+V1": 4, "I+II+V1": 4, "I+II+V2": 4,
          "I+II+V1+V5": 5, "12": 10}

CONFIG = dict(exp="exp17_wearable_configs", quest="ailab-2026-0015",
              parent_exp=["exp16_four_cfg"],
              purpose="두 평면을 한꺼번에 덮는 4전극 구성이 12유도에 얼마나 붙는가",
              change_one_thing="실험16 과 백본·분할·에폭·손실·시드공식 동일. 구성 3개만 추가",
              reuse=("실험16 의 4구성 × 3시드 × 5겹 = 60 arm 을 읽어 쓴다 — "
                     "실험16 G0-b 에서 ΔAUROC=0.0000(14/14) 로 학습 결정론성이 확인됐다"),
              configs=CONFIGS, new_configs=list(NEW_CFG), n_electrodes=N_ELEC,
              seeds=SEEDS, sites=SITE_CANDIDATES, site_plane=SITE_PLANE,
              judged_on="시드 수준 t-CI(df=2) — 실험15d 규약",
              predictions={
                  "G0": "유도 순서 항등식 + 실험16 arm 정합성",
                  "P-1": f"합집합: D({MAIN}) <= min(D(I+II), D(II+V1)) + {UNION_EPS} (전 부위)",
                  "P-2": f"배포: 전 부위에서 D({MAIN}) < {D_THR}",
                  "P-3": f"경보: {MAIN} 경보율 중앙값 <= {ALARM_R} x 12유도 경보율 중앙값",
                  "P-4": "횡단면에서 D(I+II+V2) < D(I+II+V1)",
                  "P-5": "측벽(LMI·ALMI)에서 D(I+II+V1+V5) < D(I+II+V1)"},
              caveat=("P-1 실패는 발견이다 — 실험16 에서 ILMI 가 'V1 을 더해 악화', "
                      "LMI·IMI 는 {I,II} 가 {12} 보다 나았다. 용량 경쟁이 실재한다"),
              k_fold=K_FOLD, epochs=EPOCHS, seed0=SEED0, boot=BOOT, nmin=NMIN)
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm, subprocess
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp17_wearable", CONFIG, project=PROJECT)

# ── 실험16 을 찾는다 (재사용이 전제이므로 없으면 즉시 실패)
REG = os.path.join(PROJECT, "registry.jsonl")
P16 = None
for line in (open(REG) if os.path.exists(REG) else []):
    try:
        r = json.loads(line)
    except Exception:
        continue
    if r.get("exp_id") == "exp16_four_cfg" and os.path.isdir(r.get("dir", "")):
        P16 = r["dir"]
if P16 is None:
    raise RuntimeError("registry.jsonl 에서 exp16_four_cfg 를 못 찾았습니다 — 실험16 을 먼저")
run.log(f"실험16 산출물: {P16}")

def arm_at(d, name):
    p = os.path.join(d, "arms", name, "probs.npy")
    return np.load(p) if os.path.exists(p) else None

need = [f"{c}_s{sd}_f{k}" for c in OLD_CFG for sd in SEEDS for k in range(K_FOLD)]
missing = [n for n in need if arm_at(P16, n) is None]
if missing:
    raise RuntimeError(f"실험16 arm {len(missing)}개 없음: {missing[:5]} …")
run.log(f"✅ 실험16 arm {len(need)}개 확인 — 새로 학습할 것은 "
        f"{len(NEW_CFG)}구성 × {len(SEEDS)}시드 × {K_FOLD}겹 = "
        f"{len(NEW_CFG) * len(SEEDS) * K_FOLD}회")
R16 = json.load(open(os.path.join(P16, "result.json"), encoding="utf-8"))
SITES16 = R16["sites"]
run.log(f"  실험16 부위 {SITES16}")

In [ ]:
# CELL 2 — 캐시 + 라벨 + 【G0】 유도 순서·정합성
import pandas as pd, subprocess

PTB = "/content/ptbxl"
os.makedirs(PTB, exist_ok=True)
d = os.path.join(PTB, "ptbxl_database.csv")
if not (os.path.exists(d) and os.path.getsize(d) > 0):
    subprocess.run(["wget", "-q", "-O", d,
                    "https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv"])
df = pd.read_csv(d, index_col="ecg_id")

CACHE = run.data("ptbxl_12lead_all.npz")
if not os.path.exists(CACHE):
    raise RuntimeError(f"전량 캐시가 없습니다: {CACHE} — 실험15 CELL 2 를 먼저")
z = np.load(CACHE, allow_pickle=True)
X, FOLD10, EID = z["X"], z["fold"], z["eid"]
CV = (FOLD10 - 1) % K_FOLD

# 【G0-a】 채널 순서를 신호로 확인 — V2(7)·V5(10) 을 새로 쓰므로 더 중요해졌다
lead_chk = assert_lead_order(X)
run.log("【G0-a】 유도 순서 검증 통과 — 상대오차 "
        + " · ".join(f"{k} {v:.4f}" for k, v in lead_chk["rel_err"].items()))
run.log("  → [0,1]=I,II · [6]=V1 · [7]=V2 · [10]=V5 확정")

dfa = df.loc[EID]
dfa["codes"] = dfa.scp_codes.apply(lambda s: sorted(ast.literal_eval(s).keys()))
vocab = {c for cs in dfa.codes for c in cs}
counts = {s: int(sum(s in cs for cs in dfa.codes)) for s in SITE_CANDIDATES}
assert_label_vocab(SITE_CANDIDATES, vocab, kind="scp 코드", counts=counts, min_count=NMIN)
SITES = [s for s in SITE_CANDIDATES if counts[s] >= NMIN]
NS = len(SITES)
# 【G0-b】 실험16 arm 을 읽어 쓰려면 **부위 순서까지** 같아야 한다
if SITES != SITES16:
    raise RuntimeError(f"부위 목록/순서가 실험16과 다릅니다: {SITES} vs {SITES16}")
Ymul = np.stack([[s in c for s in SITES] for c in dfa.codes]).astype("float32")
run.log(f"【G0-b】 부위 정합 ✅ {NS}개 · X{X.shape}")

FRONT = [s for s in SITES if SITE_PLANE[s] == "전두면"]
TRANS = [s for s in SITES if SITE_PLANE[s] == "횡단면"]
LAT   = [s for s in SITES if s in ("LMI", "ALMI")]     # P-5 대상(측벽 성분)
run.log(f"평면: 전두면 {FRONT} · 횡단면 {TRANS} · 측벽(P-5) {LAT}")

MASKS = {}
for c, idx in CONFIGS.items():
    m = np.zeros(12, "float32"); m[idx] = 1.0
    MASKS[c] = m
run.log("구성: " + " · ".join(f"{c}({int(MASKS[c].sum())}유도/{N_ELEC[c]}전극)"
                              for c in CONFIGS))

In [ ]:
# CELL 3 — 새 구성 3개만 학습 (45회) · 실험16 arm 은 읽어 쓴다
import tensorflow as tf
from tensorflow.keras import layers, models

def build_head(seed):
    """★ 실험15·16 과 완전히 동일."""
    tf.keras.utils.set_random_seed(seed)
    si = layers.Input((X.shape[1], 12))
    x = si
    for f, k in ((32, 9), (64, 7), (128, 5), (128, 3)):
        x = layers.Conv1D(f, k, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling1D(2)(x)
    h = layers.Dense(64, activation="relu")(layers.GlobalAveragePooling1D()(x))
    h = layers.Dropout(0.3)(h); h = layers.Dense(64, activation="relu")(h)
    m = models.Model(si, layers.Dense(NS, activation="sigmoid")(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss="binary_crossentropy")
    return m

def split(k):
    """★ 실험15·16 과 동일. 시드 무관, 겹만으로 결정."""
    te = np.where(CV == k)[0]; rest = np.where(CV != k)[0]
    rs_ = np.random.RandomState(SEED0 + k); rs_.shuffle(rest)
    n_val = max(int(len(rest) * 0.12), 200)
    return te, rest[:n_val], rest[n_val:]

def train_one(c, sd, k, save=True):
    """★ 실험15·16 과 완전히 동일한 학습 1회. (te, 예측) 을 돌려준다."""
    te, va, tr = split(k)
    mk = MASKS[c]
    m = build_head(SEED0 + 100 * k + 15 + sd)        # ★ 15·16 과 같은 시드 공식
    m.fit(X[tr] * mk, Ymul[tr], validation_data=(X[va] * mk, Ymul[va]),
          epochs=EPOCHS, batch_size=128, verbose=0)
    p = m.predict(X[te] * mk, batch_size=512, verbose=0)
    if save and k == 0 and sd == 0:
        run.save_model(m, f"head_{c}")
    tf.keras.backend.clear_session()
    return te, p

OOF = {c: {sd: np.zeros((len(EID), NS), "float32") for sd in SEEDS} for c in CONFIGS}

# ── 실험16 재사용
for c in OLD_CFG:
    for sd in SEEDS:
        for k in range(K_FOLD):
            te = np.where(CV == k)[0]
            a = arm_at(P16, f"{c}_s{sd}_f{k}")
            assert_arm_shape(a, len(te), name=f"exp16:{c}_s{sd}_f{k}")
            OOF[c][sd][te] = a
run.log(f"⏭ 실험16 arm {len(OLD_CFG) * len(SEEDS) * K_FOLD}개 읽음")

# ── 【G0-c】 재사용해도 되는가 — {I,II} 시드0 를 다시 학습해 대조 (5회)
#    실험16 은 이 검사를 통과했지만(ΔAUROC=0.0000) 그건 **그 세션의 GPU** 얘기다.
#    GPU 나 라이브러리가 바뀌면 재사용한 팔만 다른 환경이 되어 P-1 이 오염된다.
from sklearn.metrics import roc_auc_score as _auc
run.log("\n【G0-c】 재사용 정합성 — {I,II} 시드0 재학습 대조 (5회)")
_mf = os.path.join(run.dir, "manifest.json")
_gpu = json.load(open(_mf))["gpu"] if os.path.exists(_mf) else "?"
run.log(f"  이번 세션 GPU: {_gpu}")
_gpu16 = "?"
_mf16 = os.path.join(P16, "manifest.json")
if os.path.exists(_mf16):
    _gpu16 = json.load(open(_mf16)).get("gpu", "?")
run.log(f"  실험16 GPU  : {_gpu16}"
        + ("   (같음)" if _gpu16 == _gpu else "   ⚠️ 다르다 — 아래 대조가 중요해진다"))
_chk = np.zeros((len(EID), NS), "float32")
_t = time.time()
for k in range(K_FOLD):
    te, p = train_one("I+II", 0, k, save=False)
    _chk[te] = p
_per = (time.time() - _t) / K_FOLD
drift = {s: float(_auc(Ymul[:, j].astype(bool), _chk[:, j])
                  - _auc(Ymul[:, j].astype(bool), OOF["I+II"][0][:, j]))
         for j, s in enumerate(SITES)}
DRIFT_MAX = max(abs(v) for v in drift.values())
run.log("  ΔAUROC(이번−실험16) " + " ".join(f"{s}{drift[s]:+.4f}" for s in SITES))
run.log(f"  → 최대 |Δ| = {DRIFT_MAX:.4f} (문턱 {DRIFT_THR}) · 1회 학습 {_per:.0f}s")
REUSE_OK = DRIFT_MAX <= DRIFT_THR
if REUSE_OK:
    run.log("  ✅ 재사용해도 된다 — 실험16 arm 을 그대로 쓴다")
else:
    run.log(f"  ⚠️ 환경이 다르다 — 재사용한 팔이 오염된다. 조치: {ON_DRIFT}")
    if ON_DRIFT == "stop":
        raise RuntimeError(f"드리프트 {DRIFT_MAX:.4f} > {DRIFT_THR}. "
                           "ON_DRIFT='retrain' 으로 바꿔 전부 다시 학습하세요")
    if ON_DRIFT == "retrain":
        extra = len(OLD_CFG) * len(SEEDS) * K_FOLD
        run.log(f"  → 실험16 구성 {extra}회를 이 세션에서 다시 학습한다 "
                f"(추가 ≈ {extra * _per / 60:.0f}분)")

# ── 드리프트가 있으면 재사용 구성도 이 세션에서 다시 학습
TRAIN_CFG = list(NEW_CFG) if REUSE_OK or ON_DRIFT == "continue" \
    else list(OLD_CFG) + list(NEW_CFG)
t0, done, total = time.time(), 0, len(TRAIN_CFG) * len(SEEDS) * K_FOLD
run.log(f"\n새로 학습 {total}회 (예상 ≈ {total * _per / 60:.0f}분)")
for c in TRAIN_CFG:
    for sd in SEEDS:
        for k in range(K_FOLD):
            te = np.where(CV == k)[0]
            a = run.load_arm(f"{c}_s{sd}_f{k}")
            if a is None:
                te, a = train_one(c, sd, k)
                run.save_arm(f"{c}_s{sd}_f{k}", a)
            assert_arm_shape(a, len(te), name=f"{c}_s{sd}_f{k}")
            OOF[c][sd][te] = a
            done += 1
            if done == 1:
                per = time.time() - t0
                run.log(f"  ⏱ 첫 학습 {per:.0f}s → 전체 {total}회 예상 **{per*total/60:.0f}분**")
        run.log(f"  {c:<12} 시드{sd} 완료 ({done}/{total} · {time.time()-t0:.0f}s)")
run.log(f"\n총 {time.time()-t0:.0f}s")

In [ ]:
# CELL 4 — 지표: AUROC · D · 경보율 (구성 × 부위 × 시드)
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy import stats

def spec_at_sens(score, pos, target=0.90):
    p, n = score[pos], score[~pos]
    if len(p) == 0 or len(n) == 0:
        return np.nan, np.nan
    thr = float(np.quantile(p, 1.0 - target, method="lower"))
    return float((n < thr).mean()), float((score >= thr).mean())

def t_ci(vals, conf=0.95):
    v = np.asarray(vals, float); n = len(v); m = float(v.mean())
    if n < 2:
        return m, np.nan, np.nan, 0.0
    sd = float(v.std(ddof=1))
    h = float(stats.t.ppf(0.5 + conf / 2, n - 1) * sd / np.sqrt(n))
    return m, m - h, m + h, sd

ORDER = ["II", "I+II", "II+V1", "I+II+V1", "I+II+V2", "I+II+V1+V5", "12"]
M = {}
for c in ORDER:
    M[c] = {}
    for j, s in enumerate(SITES):
        y = Ymul[:, j].astype(bool)
        M[c][s] = {"auprc": [], "auroc": [], "spec": [], "alarm": []}
        for sd in SEEDS:
            sc = OOF[c][sd][:, j]
            M[c][s]["auprc"].append(float(average_precision_score(y, sc)))
            M[c][s]["auroc"].append(float(roc_auc_score(y, sc)))
            sp, al = spec_at_sens(sc, y)
            M[c][s]["spec"].append(sp); M[c][s]["alarm"].append(al)
D = {c: {s: [M[REF][s]["auroc"][i] - M[c][s]["auroc"][i] for i in range(len(SEEDS))]
         for s in SITES} for c in ORDER}

run.log("\n" + "=" * 126)
run.log("【구성별 AUROC】 시드 3개 평균 · 괄호는 전극 수")
run.log("=" * 126)
run.log(f"  {'부위':<7}{'평면':<7}{'n':>6}"
        + "".join(f"{c + f'({N_ELEC[c]})':>16}" for c in ORDER))
for j, s in enumerate(SITES):
    run.log(f"  {s:<7}{SITE_PLANE[s]:<7}{int(Ymul[:, j].sum()):>6,}"
            + "".join(f"{np.mean(M[c][s]['auroc']):>16.4f}" for c in ORDER))

run.log("\n【잔여 결손 D = AUROC(12) − AUROC(구성)】 작을수록 12유도에 가깝다")
run.log(f"  {'부위':<7}" + "".join(f"{c:>16}" for c in ORDER[:-1]))
for s in SITES:
    run.log(f"  {s:<7}" + "".join(f"{np.mean(D[c][s]):>+16.4f}" for c in ORDER[:-1]))

run.log("\n【경보율】 민감도 0.90 에서 양성 경보가 뜨는 비율 — 배포 부담")
run.log(f"  {'부위':<7}" + "".join(f"{c:>16}" for c in ORDER))
for s in SITES:
    run.log(f"  {s:<7}" + "".join(f"{np.nanmean(M[c][s]['alarm']):>16.3f}" for c in ORDER))

run.log("\n【붕괴 감시】 AUPRC < 유병률 × 1.2")
prev = {s: float(Ymul[:, j].mean()) for j, s in enumerate(SITES)}
FATAL, dead_any = False, set()
for c in ORDER:
    r = collapse_report({s: np.mean(M[c][s]["auprc"]) for s in SITES}, prev, SITES)
    dead_any |= set(r["dead"]); FATAL = FATAL or r["fatal_majority"]
    run.log(f"  {c:<12} 사망 {r['dead'] or '없음'}"
            + ("  ⛔ 과반" if r["fatal_majority"] else ""))
ALIVE = [s for s in SITES if s not in dead_any]
if FATAL:
    run.log("  ⛔ 과반 사망 — 아래 판정 무효")

In [ ]:
# CELL 5 — 사전등록 채점 (시드 수준 t-CI)
run.log("\n" + "=" * 100)
run.log("【사전등록 채점】  판정 기준 = 시드 수준 t-CI(df=2)")
run.log("=" * 100)
V = {}

# ── P-1 합집합: 두 평면을 합치는가, 맞바꾸는가
run.log(f"\n  P-1 합집합 — D({MAIN}) <= min(D(I+II), D(II+V1)) + {UNION_EPS}")
run.log(f"      {'부위':<7}{'D(I+II)':>10}{'D(II+V1)':>11}{'min':>9}"
        f"{'D(' + MAIN + ')':>13}{'초과':>9}")
u_ok, u_bad = 0, []
for s in SITES:
    a, b = np.mean(D["I+II"][s]), np.mean(D["II+V1"][s])
    mn, cur = min(a, b), np.mean(D[MAIN][s])
    exc = cur - mn
    good = exc <= UNION_EPS
    u_ok += good
    if not good:
        u_bad.append(f"{s}(+{exc:.4f})")
    run.log(f"      {s:<7}{a:>+10.4f}{b:>+11.4f}{mn:>+9.4f}{cur:>+13.4f}"
            f"{exc:>+9.4f}  {'✅' if good else '❌'}")
V["P-1"] = bool(u_ok == len(SITES))
run.log(f"  P-1 → {MARK[V['P-1']]}  {u_ok}/{len(SITES)} 부위")
if u_bad:
    run.log(f"      위반 {u_bad}")
    run.log("      ※ 두 평면을 합치지 못하고 **맞바꾸고 있다** = 용량 경쟁. "
            "유도를 더 붙일 게 아니라 구조를 바꿔야 한다(실험18)")

# ── P-2 ★★ 배포 관문 (AUROC)
run.log(f"\n  P-2 배포 관문 — 전 부위에서 D({MAIN}) < {D_THR}")
p2, worst = [], (None, -9)
for s in SITES:
    m, lo, hi, sd = t_ci(D[MAIN][s])
    v = decide(lo, hi, D_THR, "<")
    p2.append(v)
    if m > worst[1]:
        worst = (s, m)
    run.log(f"      {s:<7}{SITE_PLANE[s]:<7}D = {m:>+8.4f} [{lo:+.4f}, {hi:+.4f}] "
            f"(시드SD {sd:.4f}) → {MARK[v]}")
V["P-2"] = (True if all(x is True for x in p2)
            else False if any(x is False for x in p2) else None)
run.log(f"  P-2 → {MARK[V['P-2']]}  최악 부위 {worst[0]} D={worst[1]:+.4f}")

# ── P-3 ★ 경보 관문 (실험16 의 교훈: AUROC 가 붙어도 경보율은 벌어진다)
run.log(f"\n  P-3 경보 관문 — {MAIN} 경보율 중앙값 <= {ALARM_R} × 12유도")
run.log(f"      {'부위':<7}{MAIN:>14}{'12유도':>10}{'배수':>9}")
ratio = []
for s in SITES:
    a, b = np.nanmean(M[MAIN][s]["alarm"]), np.nanmean(M[REF][s]["alarm"])
    r = a / b if b > 0 else float("inf")
    ratio.append(r)
    run.log(f"      {s:<7}{a:>14.3f}{b:>10.3f}{r:>9.2f}배")
def _ratio(i):
    """시드 i 에서 부위별 경보율 배수의 중앙값. 12유도 경보율이 0 인 칸은 뺀다."""
    v = [M[MAIN][s]["alarm"][i] / M[REF][s]["alarm"][i]
         for s in SITES
         if M[REF][s]["alarm"][i] and np.isfinite(M[MAIN][s]["alarm"][i])]
    return float(np.median(v)) if v else float("nan")

med_r = [_ratio(i) for i in range(len(SEEDS))]
if not np.all(np.isfinite(med_r)):
    raise RuntimeError(f"경보율 배수 계산 실패 {med_r} — 12유도 경보율이 0 인 시드가 있다")
m5, lo5, hi5, sd5 = t_ci(med_r)
V["P-3"] = decide(lo5, hi5, ALARM_R, "<")
run.log(f"  P-3 → 경보율 배수 중앙값 {m5:.3f} [{lo5:.3f}, {hi5:.3f}] "
        f"vs 문턱 {ALARM_R} → {MARK[V['P-3']]}")

# ── P-4 V1 vs V2 (횡단면)
run.log("\n  P-4 전벽 커버 유도 — V2 가 V1 보다 나은가 (횡단면)")
d4 = [np.mean([D["I+II+V2"][s][i] - D["I+II+V1"][s][i] for s in TRANS])
      for i in range(len(SEEDS))]
m4, lo4, hi4, sd4 = t_ci(d4)
V["P-4"] = decide(lo4, hi4, 0.0, "<")
for s in TRANS:
    run.log(f"      {s:<7}D(V1) {np.mean(D['I+II+V1'][s]):+.4f} → "
            f"D(V2) {np.mean(D['I+II+V2'][s]):+.4f}")
run.log(f"  P-4 → ΔD(V2−V1) = {m4:+.4f} [{lo4:+.4f}, {hi4:+.4f}] → {MARK[V['P-4']]}"
        + ("   → 전벽 커버는 V2 로" if V["P-4"] else "   → V1 유지"))

# ── P-5 5번째 전극(V5)의 값어치 — 측벽
run.log("\n  P-5 5번째 전극(V5) — 측벽에서 값어치가 있는가")
d5 = [np.mean([D["I+II+V1+V5"][s][i] - D["I+II+V1"][s][i] for s in LAT])
      for i in range(len(SEEDS))] if LAT else [0.0] * len(SEEDS)
m6, lo6, hi6, sd6 = t_ci(d5)
V["P-5"] = decide(lo6, hi6, 0.0, "<")
for s in LAT:
    run.log(f"      {s:<7}D(4전극) {np.mean(D['I+II+V1'][s]):+.4f} → "
            f"D(5전극) {np.mean(D['I+II+V1+V5'][s]):+.4f}")
run.log(f"  P-5 → ΔD(+V5) = {m6:+.4f} [{lo6:+.4f}, {hi6:+.4f}] → {MARK[V['P-5']]}"
        + ("   → 5번째 전극 값어치 있음" if V["P-5"] else "   → 4전극에서 멈춘다"))

# ── 효과 vs 시드 잡음 (규약 ②)
run.log("\n  【효과 vs 시드 잡음】")
for nm, mm, ss in (("P-4 V2−V1", m4, sd4), ("P-5 +V5", m6, sd6)):
    r = abs(mm) / ss if ss > 0 else float("inf")
    run.log(f"      {nm:<12} |효과| {abs(mm):.4f} / 시드SD {ss:.4f} = {r:.2f}배"
            + ("" if r >= 1 else "  ⚠️ 잡음 이하 — 검출 불가로 종결(규약 ②)"))

run.log("\n" + "=" * 100)
for k in ("P-1", "P-2", "P-3", "P-4", "P-5"):
    run.log(f"  {k}: {MARK[V.get(k)]}")
if FATAL:
    run.log("  ⛔ 붕괴 과반 — 판정 전부 무효")
run.log("=" * 100)

In [ ]:
# CELL 6 — 그림: 전극 수 대비 결손 · 경보율 · V1/V2
import matplotlib.pyplot as plt

COLR = {"전두면": "tab:blue", "횡단면": "tab:red", "혼합": "0.6"}
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.8))

# (1) 전극 수 → 결손. 배포 관문선과 함께
for s in SITES:
    xs = [N_ELEC[c] for c in ORDER[:-1]]
    ys = [np.mean(D[c][s]) for c in ORDER[:-1]]
    o = np.argsort(xs)
    ax[0].plot(np.array(xs)[o], np.array(ys)[o], marker="o", alpha=.85,
               color=COLR[SITE_PLANE[s]],
               lw=2.2 if SITE_PLANE[s] != "혼합" else 1.0, label=s)
ax[0].axhline(0, color="k", lw=1)
ax[0].axhline(D_THR, color="green", ls="--", lw=1.4)
ax[0].text(2.1, D_THR + .004, f"배포 관문 {D_THR}", color="green", fontsize=8)
ax[0].set_xlabel("전극 수"); ax[0].set_ylabel("잔여 결손 D")
ax[0].set_title("전극 수 대비 결손"); ax[0].legend(fontsize=7, ncol=2)

# (2) 경보율 — 배포에서 실제로 아픈 지표
xs = np.arange(len(SITES)); w = 0.38
ax[1].bar(xs - w / 2, [np.nanmean(M[MAIN][s]["alarm"]) for s in SITES], w,
          label=f"{MAIN} ({N_ELEC[MAIN]}전극)", color="tab:orange")
ax[1].bar(xs + w / 2, [np.nanmean(M[REF][s]["alarm"]) for s in SITES], w,
          label="12유도 (10전극)", color="0.5")
ax[1].set_xticks(xs); ax[1].set_xticklabels(SITES, rotation=45, ha="right")
ax[1].set_ylabel("경보율 (민감도 0.90)")
ax[1].set_title(f"배포 부담 · P-3 {MARK[V['P-3']]}"); ax[1].legend(fontsize=8)

# (3) V1 vs V2 — 전벽 커버 유도 결정
ax[2].bar(xs - w / 2, [np.mean(D["I+II+V1"][s]) for s in SITES], w,
          label="I+II+V1", color="tab:purple")
ax[2].bar(xs + w / 2, [np.mean(D["I+II+V2"][s]) for s in SITES], w,
          label="I+II+V2", color="tab:green")
ax[2].axhline(0, color="k", lw=1)
ax[2].set_xticks(xs); ax[2].set_xticklabels(SITES, rotation=45, ha="right")
ax[2].set_ylabel("잔여 결손 D")
ax[2].set_title(f"전벽 커버 유도 · P-4 {MARK[V['P-4']]}"); ax[2].legend(fontsize=8)

plt.tight_layout(); run.save_fig("wearable_configs", fig); plt.show()

In [ ]:
# CELL 7 — 결과 저장
res = {
    "week": 2, "exp_id": "exp17_wearable", "quest": "ailab-2026-0015",
    "task": "4전극 웨어러블 — 두 평면을 한 번에 덮으면 12유도에 얼마나 붙나",
    "split": "inter", "step": "exp17-wearable-configs",
    "metric": f"max_D_{MAIN}",
    "value": round(float(max(np.mean(D[MAIN][s]) for s in SITES)), 4),
    "passed": bool(V.get("P-2") is True and V.get("P-3") is True and not FATAL),
    "date": time.strftime("%Y-%m-%d"),
    "k_fold": K_FOLD, "n_seeds": len(SEEDS), "configs": ORDER,
    "n_electrodes": N_ELEC, "sites": SITES, "front": FRONT, "trans": TRANS,
    "main_config": MAIN, "collapse_fatal": FATAL, "alive": ALIVE,
    "reuse_ok": bool(REUSE_OK), "drift_max_abs": round(float(DRIFT_MAX), 4),
    "drift_by_site": {s: round(v, 4) for s, v in drift.items()},
    "sec_per_training": round(float(_per), 1),
    "reused_arms": 0 if not REUSE_OK and ON_DRIFT == "retrain"
                   else len(OLD_CFG) * len(SEEDS) * K_FOLD,
    "trained_arms": len(TRAIN_CFG) * len(SEEDS) * K_FOLD + K_FOLD,
    "P-1": V.get("P-1"), "P-2": V.get("P-2"), "P-3": V.get("P-3"),
    "P-4": V.get("P-4"), "P-5": V.get("P-5"),
    "p1_violations": u_bad,
    "p2_worst_site": worst[0], "p2_worst_D": round(float(worst[1]), 4),
    "p3_alarm_ratio_median": round(float(m5), 3),
    "p3_ci": [round(float(lo5), 3), round(float(hi5), 3)],
    "p4_v2_minus_v1": round(float(m4), 4),
    "p4_ci": [round(float(lo4), 4), round(float(hi4), 4)],
    "p5_v5_gain": round(float(m6), 4),
    "p5_ci": [round(float(lo6), 4), round(float(hi6), 4)],
    "auroc_mean": {c: {s: round(float(np.mean(M[c][s]["auroc"])), 4) for s in SITES}
                   for c in ORDER},
    "D_mean": {c: {s: round(float(np.mean(D[c][s])), 4) for s in SITES} for c in ORDER},
    "alarm_rate": {c: {s: round(float(np.nanmean(M[c][s]["alarm"])), 4) for s in SITES}
                   for c in ORDER},
    "spec_at_sens90": {c: {s: round(float(np.nanmean(M[c][s]["spec"])), 4) for s in SITES}
                       for c in ORDER},
    "verdict": " · ".join(f"{k} {MARK[V.get(k)]}" for k in
                          ("P-1", "P-2", "P-3", "P-4", "P-5")),
}
res["summary"] = (f"{MAIN}({N_ELEC[MAIN]}전극) 최악 결손 {res['value']:+.4f}({worst[0]}) · "
                  f"경보율 배수 중앙값 {m5:.2f} · " + res["verdict"])
run.save_json("result.json", res)
run.log("\n" + json.dumps({k: res[k] for k in
                           ("metric", "value", "passed", "P-1", "P-2", "P-3", "P-4",
                            "P-5", "summary")}, ensure_ascii=False, indent=2))
run.finish(res)